In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("foodAllergyAnalysisZenodo.csv")

print(df.shape)
df.info()

(333200, 50)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 333200 entries, 0 to 333199
Data columns (total 50 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   SUBJECT_ID               333200 non-null  int64  
 1   BIRTH_YEAR               333200 non-null  int64  
 2   GENDER_FACTOR            333200 non-null  object 
 3   RACE_FACTOR              333200 non-null  object 
 4   ETHNICITY_FACTOR         333200 non-null  object 
 5   PAYER_FACTOR             333200 non-null  object 
 6   ATOPIC_MARCH_COHORT      333200 non-null  bool   
 7   AGE_START_YEARS          333200 non-null  float64
 8   AGE_END_YEARS            333200 non-null  float64
 9   SHELLFISH_ALG_START      5246 non-null    float64
 10  SHELLFISH_ALG_END        1051 non-null    float64
 11  FISH_ALG_START           1796 non-null    float64
 12  FISH_ALG_END             527 non-null     float64
 13  MILK_ALG_START           7289 non-null    floa

In [3]:
df.head()

,SUBJECT_ID,BIRTH_YEAR,GENDER_FACTOR,RACE_FACTOR,ETHNICITY_FACTOR,PAYER_FACTOR,ATOPIC_MARCH_COHORT,AGE_START_YEARS,AGE_END_YEARS,SHELLFISH_ALG_START,...,CASHEW_ALG_END,ATOPIC_DERM_START,ATOPIC_DERM_END,ALLERGIC_RHINITIS_START,ALLERGIC_RHINITIS_END,ASTHMA_START,ASTHMA_END,FIRST_ASTHMARX,LAST_ASTHMARX,NUM_ASTHMARX
0,1,2006,S1 - Female,R1 - Black,E0 - Non-Hispanic,P1 - Medicaid,False,0.093087,3.164956,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1994,S1 - Female,R0 - White,E0 - Non-Hispanic,P0 - Non-Medicaid,False,12.232717,18.880219,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.262834,18.880219,2.0
2,3,2006,S0 - Male,R0 - White,E1 - Hispanic,P0 - Non-Medicaid,True,0.010951,6.726899,NaN,...,NaN,4.884326,NaN,3.917864,6.157426,5.127995,NaN,1.404517,6.157426,4.0
3,4,2004,S0 - Male,R4 - Unknown,E1 - Hispanic,P0 - Non-Medicaid,False,2.398357,9.111567,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2006,S1 - Female,R1 - Black,E0 - Non-Hispanic,P0 - Non-Medicaid,False,0.013689,6.193018,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df["PEANUT_ALLERGY"] = df["PEANUT_ALG_START"].notna().astype(int)
df["MILK_ALLERGY"] = df["MILK_ALG_START"].notna().astype(int)
df["EGG_ALLERGY"] = df["EGG_ALG_START"].notna().astype(int)
df["ASTHMA_PRESENT"] = df["ASTHMA_START"].notna().astype(int)
df["ECZEMA_PRESENT"] = df["ATOPIC_DERM_START"].notna().astype(int)

print(df["PEANUT_ALLERGY"].value_counts())
print(df["MILK_ALLERGY"].value_counts())
print(df["EGG_ALLERGY"].value_counts())
print(df["ASTHMA_PRESENT"].value_counts())
print(df["ECZEMA_PRESENT"].value_counts())

PEANUT_ALLERGY
0    324547
1      8653
Name: count, dtype: int64
MILK_ALLERGY
0    325911
1      7289
Name: count, dtype: int64
EGG_ALLERGY
0    327135
1      6065
Name: count, dtype: int64
ASTHMA_PRESENT
0    269326
1     63874
Name: count, dtype: int64
ECZEMA_PRESENT
0    283685
1     49515
Name: count, dtype: int64


In [5]:
# define features and targets
features = [
    "AGE_START_YEARS",
    "GENDER_FACTOR",
    "RACE_FACTOR",
    "ETHNICITY_FACTOR",
    "ASTHMA_PRESENT",
    "ECZEMA_PRESENT"
]

target = [
    "PEANUT_ALLERGY",
    "MILK_ALLERGY",
    "EGG_ALLERGY"
]

data = df[features + target].copy()

data = data.dropna()
print("Clean dataset shape:", data.shape)
data = pd.get_dummies(data, drop_first=True)
          

Clean dataset shape: (333200, 9)


In [6]:
# train/test split
X = data.drop(target, axis=1)
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [7]:
# training models
base_model = LogisticRegression(max_iter=1000)

model = MultiOutputClassifier(base_model)
model.fit(X_train, y_train)

pred_probs = model.predict_proba(X_test)

peanut_auc = roc_auc_score(y_test["PEANUT_ALLERGY"], pred_probs[0][:,1])
milk_auc = roc_auc_score(y_test["MILK_ALLERGY"], pred_probs[1][:,1])
egg_auc = roc_auc_score(y_test["EGG_ALLERGY"], pred_probs[2][:,1])

print("Peanut Allergy ROC-AUC:", peanut_auc)
print("Milk Allergy ROC-AUC:", milk_auc)
print("Egg Allergy ROC-AUC:", egg_auc)

Peanut Allergy ROC-AUC: 0.7469678515230813
Milk Allergy ROC-AUC: 0.7206569578589385
Egg Allergy ROC-AUC: 0.7849413224938987


In [8]:
# prediction function
def predict_allergies(age, gender, race, ethnicity, asthma, eczema):

    input_data = pd.DataFrame({
        "AGE_START_YEARS": [age],
        "GENDER_FACTOR": [gender],
        "RACE_FACTOR": [race],
        "ETHNICITY_FACTOR": [ethnicity],
        "ASTHMA_PRESENT": [asthma],
        "ECZEMA_PRESENT": [eczema]
    })

    # apply same encoding used in training
    input_data = pd.get_dummies(input_data)

    # align columns with training data
    input_data = input_data.reindex(columns=X_train.columns, fill_value=0)

    probs = model.predict_proba(input_data)

    result = {
    "peanut_allergy_risk": float(probs[0][0][1]),
    "milk_allergy_risk": float(probs[1][0][1]),
    "egg_allergy_risk": float(probs[2][0][1])
    }

    return result

In [9]:
predict_allergies(20,2,3,2,0,0)

{'peanut_allergy_risk': 0.006273547727097787,
 'milk_allergy_risk': 0.0014026828510703558,
 'egg_allergy_risk': 0.000904323294619445}